# 08 — Error Analysis

Tabel metrik memberi tahu *berapa banyak* yang salah; notebook ini mencari **apa yang salah dan mengapa**:

1. Confusion matrix ternormalisasi ketiga model berdampingan
2. Tabel metrik per kelas
3. Pola kesalahan: kelas X paling sering tertukar dengan Y
4. Galeri citra biji yang salah diklasifikasi
5. Analisis confidence: apakah model tahu saat dirinya ragu?

Semua berjalan di atas `results/analysis/<model>_test_predictions.csv` yang dihasilkan `scripts.evaluate` (atau sel terakhir notebook 03-05).

## Setup

In [ ]:
# Bootstrap: tambahkan root proyek ke sys.path agar modul src/ & configs/ dapat diimpor.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from configs.base_config import ANALYSIS_DIR, CLASS_NAMES, FIGURES_DIR
from configs.cnn_config import CNN_CONFIG
from src.analysis.confidence_analysis import (
    accuracy_vs_threshold,
    confidence_summary,
    identify_uncertain_predictions,
    plot_confidence_distribution,
    plot_confidence_per_class,
)
from src.analysis.confusion_analysis import (
    error_pattern_analysis,
    generate_classification_report_table,
    plot_confusion_matrices_comparison,
    plot_error_pattern_heatmap,
)
from src.analysis.error_analysis import (
    error_summary_by_class,
    plot_class_gallery,
    plot_error_gallery,
)
from src.analysis.style import apply_plot_style
from src.evaluation.predict import load_predictions

apply_plot_style()

## 1. Muat tabel prediksi

In [ ]:
tables = {}
for name in CNN_CONFIG:
    path = ANALYSIS_DIR / f'{name}_test_predictions.csv'
    if path.exists():
        tables[name] = load_predictions(path)
        print(f'{name}: {len(tables[name]):,} baris')
    else:
        print(f'{name}: {path.name} belum ada')

if not tables:
    raise FileNotFoundError(
        'Belum ada CSV prediksi. Jalankan: python -m scripts.evaluate --model <m> --checkpoint <p>'
    )

accuracy = {n: float(t['correct'].mean()) for n, t in tables.items()}
BEST_MODEL = max(accuracy, key=accuracy.get)
print('\nakurasi:', {k: round(v, 4) for k, v in accuracy.items()})
print('model terbaik (akurasi):', BEST_MODEL)

## 2. Confusion matrix berdampingan

In [ ]:
pairs = {n: (t['y_true'].tolist(), t['y_pred'].tolist()) for n, t in tables.items()}
plot_confusion_matrices_comparison(
    pairs,
    class_names=CLASS_NAMES,
    normalize='true',
    output_path=FIGURES_DIR / 'confusion_matrices_comparison.png',
)
plt.show()

## 3. Metrik per kelas, semua model

In [ ]:
frames = []
for name, (y_true, y_pred) in pairs.items():
    table = generate_classification_report_table(y_true, y_pred, CLASS_NAMES)
    table.insert(0, 'model', name)
    frames.append(table)

df_per_class = pd.concat(frames)
df_per_class.to_csv(ANALYSIS_DIR / 'per_class_metrics.csv', encoding='utf-8')
df_per_class

## 4. Pola kesalahan

`share_of_true` lebih berguna daripada jumlah absolut: berapa **persen** biji kelas X yang bocor ke kelas Y.

In [ ]:
for name, (y_true, y_pred) in pairs.items():
    patterns = error_pattern_analysis(y_true, y_pred, CLASS_NAMES, top_n=8)
    print(f'=== {name} ===')
    print(patterns.to_string(index=False) if not patterns.empty else '  tanpa kesalahan')
    print()

In [ ]:
plot_error_pattern_heatmap(
    *pairs[BEST_MODEL],
    class_names=CLASS_NAMES,
    output_path=FIGURES_DIR / f'{BEST_MODEL}_error_pattern.png',
    title=f'Pola kesalahan - {BEST_MODEL}',
)
plt.show()

## 5. Ringkasan error per kelas

In [ ]:
summaries = []
for name, frame in tables.items():
    summary = error_summary_by_class(frame, class_names=CLASS_NAMES)
    summary.insert(0, 'model', name)
    summaries.append(summary)

pd.concat(summaries)

## 6. Galeri biji yang salah diklasifikasi

Diurutkan dari confidence **tertinggi**: kesalahan yang model yakini justru paling informatif. Perhatikan apakah ada crop yang sebenarnya labelnya keliru saat anotasi — itu temuan tentang datanya, bukan tentang modelnya.

In [ ]:
N_SAMPLES = 20

frame = tables[BEST_MODEL]
wrong = frame.loc[~frame['correct']].sort_values('confidence', ascending=False)
print(f'{len(wrong):,} prediksi salah dari {len(frame):,} ({len(wrong)/len(frame)*100:.1f}%)')

plot_error_gallery(
    wrong.head(N_SAMPLES).reset_index(drop=True),
    n_cols=5,
    output_path=FIGURES_DIR / f'{BEST_MODEL}_error_gallery.png',
    title=f'Kesalahan paling percaya diri - {BEST_MODEL}',
)
plt.show()

## 7. Pembanding: contoh prediksi yang benar per kelas

In [ ]:
plot_class_gallery(
    frame,
    class_names=CLASS_NAMES,
    n_per_class=5,
    only_correct=True,
    output_path=FIGURES_DIR / f'{BEST_MODEL}_class_gallery.png',
    title=f'Prediksi benar per kelas - {BEST_MODEL}',
)
plt.show()

## 8. Analisis confidence

`confidence_gap` = rata-rata confidence saat benar dikurangi saat salah. Makin besar, makin bisa diandalkan sebagai sinyal "model sedang ragu".

In [ ]:
rows = []
for name, t in tables.items():
    summary = confidence_summary(t)
    summary['model'] = name
    rows.append(summary)

pd.DataFrame(rows).set_index('model')

In [ ]:
plot_confidence_distribution(
    frame,
    output_path=FIGURES_DIR / f'{BEST_MODEL}_confidence_distribution.png',
    title=f'Distribusi confidence - {BEST_MODEL}',
)
plt.show()

plot_confidence_per_class(
    frame,
    class_names=CLASS_NAMES,
    output_path=FIGURES_DIR / f'{BEST_MODEL}_confidence_per_class.png',
    title=f'Confidence per kelas - {BEST_MODEL}',
)
plt.show()

## 9. Simulasi *human-in-the-loop*

Bila biji ber-confidence rendah dilimpahkan ke pemeriksa manusia, berapa akurasi yang tersisa untuk yang otomatis, dan berapa banyak yang harus diperiksa? Ini cara menerjemahkan metrik jadi keputusan operasional.

In [ ]:
table = accuracy_vs_threshold(frame)
display(table)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(table['threshold'], table['accuracy_on_covered'], 'o-', label='akurasi (otomatis)')
ax.plot(table['threshold'], table['coverage'], 's--', label='cakupan otomatis')
ax.set_xlabel('ambang confidence')
ax.set_ylabel('proporsi')
ax.set_title(f'Trade-off ambang confidence - {BEST_MODEL}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
uncertain = identify_uncertain_predictions(frame, threshold=0.5)
print(f'{len(uncertain):,} biji dengan confidence < 0.5')
uncertain.head(15)[['true_label', 'pred_label', 'confidence']]

## 10. Catatan temuan (isi setelah melihat hasilnya)

- Kelas apa yang paling sering tertukar, dan apakah kemiripannya masuk akal   secara visual?
- Berapa biji cacat yang lolos sebagai `good`? Ini kesalahan termahal untuk   quality control.
- Apakah ada crop yang labelnya memang keliru saat anotasi?
- Apakah pola kesalahan ketiga arsitektur serupa? Kalau ya, penyebabnya lebih   mungkin di data daripada di arsitektur.